# 🩺 NetraAI (SIH26038) — DRIVE Retinal Blood Vessel Segmentation Suite
### Specialized Google Colab Training Notebook for DRIVE Dataset

This notebook trains an end-to-end **U-Net Neural Network** on the **DRIVE (Digital Retinal Images for Vessel Extraction)** dataset:
- **Target Task**: Pixel-accurate Retinal Vasculature & Vessel Tree Extraction
- **Architecture**: U-Net with Skip Connections and Feature Concatenation
- **Loss Function**: Combined BCE + Soft Dice Loss (`DiceBCELoss`) for extreme class imbalance
- **Outputs**: `unet_vessels.pt` (Directly loads into NetraAI `ml/checkpoints/`)
- **Execution Time**: ~5–8 minutes on Google Colab Free GPU (NVIDIA T4)

--- 
## 1. Setup Environment & Verify GPU Hardware

In [ ]:
# Check CUDA device
!nvidia-smi

import os
import sys
import glob
import zipfile
import random
import numpy as np
from PIL import Image
import cv2
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Set deterministic seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Training on device: {device}")

--- 
## 2. Mount Google Drive & Locate DRIVE Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Search candidate directories in Google Drive
candidate_paths = [
    '/content/drive/MyDrive/Datasests/DRIVE(Vessel Extraction)',
    '/content/drive/MyDrive/DRIVE(Vessel Extraction)',
    '/content/drive/MyDrive/DRIVE',
    '/content/drive/MyDrive/Datasests/DRIVE',
    '/content/DRIVE(Vessel Extraction)',
    '/content/drive/MyDrive/DRIVE.zip',
    '/content/drive/MyDrive/Datasests/DRIVE.zip'
]

DRIVE_ROOT = None
for p in candidate_paths:
    if os.path.exists(p):
        if p.endswith('.zip'):
            print(f"📦 Found zip archive at {p}. Extracting to /content/drive_data ...")
            with zipfile.ZipFile(p, 'r') as zip_ref:
                zip_ref.extractall('/content/drive_data')
            DRIVE_ROOT = '/content/drive_data'
        else:
            DRIVE_ROOT = p
        break

if DRIVE_ROOT is None:
    print("⚠️ DRIVE dataset folder not automatically found.")
    print("Please enter the exact path to your DRIVE folder below:")
    DRIVE_ROOT = input("Enter DRIVE dataset path: ").strip()

print(f"✅ DRIVE Dataset Root resolved to: {DRIVE_ROOT}")

# Verify subdirectories
train_img_dir = os.path.join(DRIVE_ROOT, 'training', 'images')
train_mask_dir = os.path.join(DRIVE_ROOT, 'training', '1st_manual')
test_img_dir = os.path.join(DRIVE_ROOT, 'test', 'images')
test_mask_dir = os.path.join(DRIVE_ROOT, 'test', '1st_manual')

print(f"Training images: {len(glob.glob(os.path.join(train_img_dir, '*.*')))}")
print(f"Training masks:  {len(glob.glob(os.path.join(train_mask_dir, '*.*')))}")
print(f"Test images:     {len(glob.glob(os.path.join(test_img_dir, '*.*')))}")

--- 
## 3. Retinal Preprocessing & Dataset Loader

Vessels appear with high contrast in the **green channel** of retinal fundus images. We apply:
1. **Green-Channel Extraction & Inversion**
2. **CLAHE (Contrast Limited Adaptive Histogram Equalization)** to amplify capillary contrast
3. **Resolution Normalization** (512x512)
4. **Data Augmentation** (Flips, 90-degree rotations, brightness jitter)

In [ ]:
def preprocess_vessel_image(bgr_image, target_size=(512, 512)):
    """
    Enhances retinal vessel tree contrast:
    Extracts green channel, applies CLAHE, and normalizes to target size.
    """
    resized = cv2.resize(bgr_image, target_size, interpolation=cv2.INTER_AREA)
    g = resized[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8))
    g_clahe = clahe.apply(g)
    
    # Stack 3-channel: [Original Green, Enhanced CLAHE, Inverted Green]
    inverted_g = cv2.bitwise_not(g_clahe)
    stacked = np.stack([g, g_clahe, inverted_g], axis=-1)
    return stacked


class DRIVEDataset(Dataset):
    def __init__(self, img_dir, mask_dir, is_training=True, target_size=(512, 512)):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.is_training = is_training
        self.target_size = target_size
        
        self.img_files = sorted(glob.glob(os.path.join(img_dir, '*.tif')) + glob.glob(os.path.join(img_dir, '*.jpg')) + glob.glob(os.path.join(img_dir, '*.png')))
        
    def __len__(self):
        return len(self.img_files)
        
    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        basename = os.path.splitext(os.path.basename(img_path))[0]
        prefix = basename.split('_')[0]
        
        # Match 1st_manual mask (e.g. '21_manual1.gif' or '21_manual1.tif' or '*.png')
        mask_candidates = [
            os.path.join(self.mask_dir, f"{prefix}_manual1.gif"),
            os.path.join(self.mask_dir, f"{prefix}_manual1.tif"),
            os.path.join(self.mask_dir, f"{prefix}_manual1.png"),
            os.path.join(self.mask_dir, f"{basename}.gif"),
            os.path.join(self.mask_dir, f"{basename}.png")
        ]
        
        mask_path = None
        for mc in mask_candidates:
            if os.path.exists(mc):
                mask_path = mc
                break
                
        # Load image
        bgr = cv2.imread(img_path)
        if bgr is None:
            bgr = np.array(Image.open(img_path))
            if bgr.ndim == 2: bgr = cv2.cvtColor(bgr, cv2.COLOR_GRAY2BGR)
            elif bgr.shape[2] == 3: bgr = cv2.cvtColor(bgr, cv2.COLOR_RGB2BGR)
            
        proc_img = preprocess_vessel_image(bgr, self.target_size)
        
        # Load ground truth binary mask
        if mask_path and os.path.exists(mask_path):
            mask_pil = Image.open(mask_path)
            mask_np = np.array(mask_pil)
            mask_resized = cv2.resize(mask_np, self.target_size, interpolation=cv2.INTER_NEAREST)
            mask_bin = (mask_resized > 127).astype(np.float32)
        else:
            mask_bin = np.zeros(self.target_size, dtype=np.float32)
            
        # Augmentation for training
        if self.is_training:
            if random.random() > 0.5:
                proc_img = np.fliplr(proc_img).copy()
                mask_bin = np.fliplr(mask_bin).copy()
            if random.random() > 0.5:
                proc_img = np.flipud(proc_img).copy()
                mask_bin = np.flipud(mask_bin).copy()
            k = random.choice([0, 1, 2, 3])
            if k > 0:
                proc_img = np.rot90(proc_img, k).copy()
                mask_bin = np.rot90(mask_bin, k).copy()
                
        # Normalize to [0, 1] tensors
        img_tensor = torch.tensor(np.transpose(proc_img, (2, 0, 1)), dtype=torch.float32) / 255.0
        mask_tensor = torch.tensor(mask_bin, dtype=torch.float32).unsqueeze(0)
        
        return img_tensor, mask_tensor

# Create Datasets and DataLoaders
train_dataset = DRIVEDataset(train_img_dir, train_mask_dir, is_training=True)
val_dataset = DRIVEDataset(test_img_dir, test_mask_dir, is_training=False)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=2)

print(f"✅ Successfully loaded {len(train_dataset)} training scans and {len(val_dataset)} validation scans!")

--- 
## 4. U-Net Neural Network Architecture

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)


class VesselUNet(nn.Module):
    """
    Standard U-Net Architecture with 4-level encoder-decoder skip connections.
    """
    def __init__(self, in_channels=3, out_channels=1):
        super().__init__()
        # Encoder
        self.enc1 = DoubleConv(in_channels, 32)
        self.enc2 = DoubleConv(32, 64)
        self.enc3 = DoubleConv(64, 128)
        self.enc4 = DoubleConv(128, 256)
        self.pool = nn.MaxPool2d(2, 2)
        
        # Bottleneck
        self.bottleneck = DoubleConv(256, 512)
        
        # Decoder
        self.up4 = nn.ConvTranspose2d(512, 256, 2, stride=2)
        self.dec4 = DoubleConv(512, 256)
        
        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.dec3 = DoubleConv(256, 128)
        
        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec2 = DoubleConv(128, 64)
        
        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec1 = DoubleConv(64, 32)
        
        # Output projection (logits)
        self.out_conv = nn.Conv2d(32, out_channels, 1)
        
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        
        b = self.bottleneck(self.pool(e4))
        
        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))
        
        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))
        
        return self.out_conv(d1)

model = VesselUNet().to(device)
print(f"✅ U-Net instantiated with {sum(p.numel() for p in model.parameters()):,} parameters.")

--- 
## 5. Dice + BCE Loss & Optimization

In [ ]:
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()
        
    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)
        probs = torch.sigmoid(logits)
        
        probs_flat = probs.view(-1)
        targets_flat = targets.view(-1)
        
        intersection = (probs_flat * targets_flat).sum()
        dice_score = (2.0 * intersection + self.smooth) / (probs_flat.sum() + targets_flat.sum() + self.smooth)
        dice_loss = 1.0 - dice_score
        
        return 0.5 * bce_loss + 0.5 * dice_loss


def compute_dice_score(preds_bin, targets):
    intersection = (preds_bin * targets).sum()
    total = preds_bin.sum() + targets.sum()
    if total == 0: return 1.0
    return float(2.0 * intersection / (total + 1e-6))


criterion = DiceBCELoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=25, eta_min=1e-5)

--- 
## 6. Model Training & Validation Loop (~5 Minutes)

In [ ]:
EPOCHS = 25
best_dice = 0.0
save_path = '/content/unet_vessels.pt'

train_losses, val_dices = [], []

print("=" * 65)
print(f"   STARTING U-NET TRAINING ON DRIVE RETINAL VESSEL DATASET")
print("=" * 65)

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    
    for images, masks in train_loader:
        images, masks = images.to(device), masks.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
    scheduler.step()
    epoch_loss = running_loss / len(train_dataset)
    train_losses.append(epoch_loss)
    
    # Validation phase
    model.eval()
    epoch_dice = 0.0
    with torch.no_grad():
        for val_images, val_masks in val_loader:
            val_images, val_masks = val_images.to(device), val_masks.to(device)
            val_outputs = model(val_images)
            probs = torch.sigmoid(val_outputs)
            preds_bin = (probs > 0.5).float()
            
            epoch_dice += compute_dice_score(preds_bin.cpu().numpy(), val_masks.cpu().numpy()) * val_images.size(0)
            
    val_dice = epoch_dice / max(1, len(val_dataset))
    val_dices.append(val_dice)
    
    is_best = val_dice > best_dice
    if is_best:
        best_dice = val_dice
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'best_dice': best_dice,
            'architecture': 'VesselUNet'
        }, save_path)
        
    print(f"Epoch [{epoch:02d}/{EPOCHS}] | Train Loss: {epoch_loss:.4f} | Val Dice: {val_dice:.4f} {'🔥 (New Best!)' if is_best else ''}")

print("=" * 65)
print(f"🎉 Training Complete! Best Validation Dice Score: {best_dice:.4f}")
print(f"💾 Best model weights saved to: {save_path}")

--- 
## 7. Visual Inspection of Segmented Vasculature

In [ ]:
# Load best weights
checkpoint = torch.load(save_path, map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Sample a validation image
sample_idx = 0
sample_img, sample_mask = val_dataset[sample_idx]

with torch.no_grad():
    inp = sample_img.unsqueeze(0).to(device)
    pred_logits = model(inp)
    pred_mask = torch.sigmoid(pred_logits).squeeze().cpu().numpy()
    pred_binary = (pred_mask > 0.5).astype(np.uint8)

# Plot comparisons
fig, axes = plt.subplots(1, 4, figsize=(18, 5))
axes[0].imshow(sample_img[0].numpy(), cmap='gray')
axes[0].set_title("Preprocessed Fundus (Green Channel)", fontsize=12)
axes[0].axis('off')

axes[1].imshow(sample_mask.squeeze().numpy(), cmap='gray')
axes[1].set_title("Ground Truth Vessel Mask", fontsize=12)
axes[1].axis('off')

axes[2].imshow(pred_mask, cmap='hot')
axes[2].set_title("U-Net Vessel Probability Heatmap", fontsize=12)
axes[2].axis('off')

axes[3].imshow(pred_binary, cmap='gray')
axes[3].set_title(f"Thresholded Binary Vessels (>0.5)", fontsize=12)
axes[3].axis('off')

plt.tight_layout()
plt.show()

--- 
## 8. Download Trained `unet_vessels.pt` Checkpoint

Run the cell below to download `unet_vessels.pt` directly to your laptop.
Place the downloaded file into:
📁 `c:\Users\LENONO\Desktop\SIH 2026\SIH26038\ml\checkpoints\unet_vessels.pt`

In [ ]:
from google.colab import files

if os.path.exists(save_path):
    print(f"📥 Downloading {save_path} to your computer...")
    files.download(save_path)
else:
    print("Error: Checkpoint file not found.")